In [1]:
import numpy as np
import pickle
import pandas as pd
import yaml
import os
from matplotlib import pyplot as plt
import pysindy as ps
from hsa_hopper.collocation import interp_covector

def savgol(t,x,deg,window,ord=[0]):
    N = x.shape[0]
    outputs = np.zeros((N-window,len(ord)))
    for i in range(N-window):
        a = t[i]
        b = t[i+window]
        c = t[i+window//2]
        tensor = np.vstack([interp_covector(t[i+j],a,b,deg+1) for j in range(window)])
        coeffs = np.linalg.lstsq(tensor,x[i:i+window],rcond=None)[0]
        for j,o in enumerate(ord):
            tensor = interp_covector(c,a,b,deg+1,ord=o)[0]
            outputs[i,j] = np.dot(tensor,coeffs)
    return outputs

def smooth_dynamics(df, deg, window):
    new_df = pd.DataFrame(columns=df.columns)
    _t = df['t'].to_numpy()
    _t = _t - _t[0]
    new_df['t'] = _t[window//2:-window//2]
    _x = df['x'].to_numpy()
    outputs = savgol(_t,_x,deg,window,ord=[0,1,2])
    new_df['x'] = outputs[:,0]
    new_df['xdot'] = outputs[:,1]
    new_df['xddot'] = outputs[:,2]
    _tau = df['tau'].to_numpy()
    outputs = savgol(_t,_tau,deg,window)
    new_df['tau'] = outputs[:,0]
    return new_df


In [2]:
from hsa_hopper.kinematics import KinematicParameters, forward_kinematics
"""
Load all data from the HSA characterization dataset
"""
characterization = {
    'folder' : r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\data\hsa_identification\2024-06-13_1718318116'
}
characterization['data'] = []
characterization['hsa_angle'] = []
keep_columns = ['motor_angle', 'motor_torque', 'times']
columns = ['x','y','l','xdot','ydot','ldot','xddot','yddot','lddot','tau','t','psi','dydx', 'dldx', 'd2yd2x', 'd2ld2x', 'mode','m']

# dictionary for remapping column names
column_map = {
    'motor_angle': lambda motor_angle: ('x',motor_angle), 
    'motor_torque': lambda motor_torque: ('tau', motor_torque),
    'times' : lambda times: ('t', times),
}

with open(os.path.join(characterization['folder'], 'experiment_config.yaml'), 'r') as f:
    characterization['experiment_config'] = yaml.load(f, yaml.Loader)

with open(os.path.join(characterization['folder'], 'hardware_config.yaml'), 'r') as f:
    characterization['hardware_config'] = yaml.load(f, yaml.Loader)

deg = 2
window = 2*(deg+1) + 3 
with open(os.path.join(characterization['folder'], 'data.pickle'), 'rb') as f:
    all_data = pickle.load(f)
    for idx in range(len(all_data['motor_angle'])):
        # input_df = pd.DataFrame({key : all_data[key][idx] for key in keep_columns})
        input_df = pd.DataFrame(columns=columns)
        data_remapped = {}
        for key, map in column_map.items():
            new_key, new_data = map(all_data[key][idx])
            input_df[new_key] = new_data
        smoothed_df = smooth_dynamics(input_df, deg, window)
        x = smoothed_df['x'].to_numpy()
        hardware_config = characterization['hardware_config']
        kin_params = KinematicParameters(**hardware_config['kinematics'])
        FK = forward_kinematics(kin_params,x,jacobian=True,hessian=True)
        smoothed_df['y'] = FK[0][0]
        smoothed_df['l'] = FK[0][1]
        smoothed_df['dydx'] = FK[1][0]
        smoothed_df['dldx'] = FK[1][1]
        smoothed_df['d2yd2x'] = FK[2][0]
        smoothed_df['d2ld2x'] = FK[2][1]
        smoothed_df['psi'] = (np.pi/180)*all_data['hsa_angle'][idx]*np.ones(len(smoothed_df))
        smoothed_df['mode'] = np.zeros(len(smoothed_df))
        smoothed_df['m'] = np.zeros(len(smoothed_df))
        characterization['data'].append(smoothed_df)

In [3]:
"""
Load all data from a hopping experiment.
"""

# constants for converting servo pulse to angle
_MIN_PULSE = 500
_MAX_PULSE = 2500
_ZERO_SETPOINT = 1500
_ANGLE_RANGE = 1800
def pulse_to_angle(pulse, gear_ratio):
    return (_ANGLE_RANGE / (_MAX_PULSE-_MIN_PULSE)) * gear_ratio * (pulse - _ZERO_SETPOINT)

# remapping data frame columns to match characterization data
column_map = {
    'x_deg': lambda x_deg: ('x',x_deg*np.pi/180), 
    # 'torque': lambda torque: ('tau', torque),
    'u_ff': lambda torque: ('tau', torque),
    't_s' : lambda t_s: ('t', t_s),
    'mode': lambda mode: ('mode', mode)
}

hop_experiment_dir = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\data\hsa_hopper_data_06_13_2024\hop_experiments\hsa'
hopping = {'data': [], 'experiment_config': [], 'hardware_config': [], 'energy': []}

# order = 2
# decimate = 5
deg = 2
window = 2*(deg+1) + 3
for folder in os.listdir(hop_experiment_dir):
    path = os.path.join(hop_experiment_dir, folder)
    with open(os.path.join(path, 'experiment_config.yaml'), 'r') as f:
        hopping['experiment_config'].append(yaml.load(f,yaml.Loader))
    with open(os.path.join(path, 'hardware_config.yaml'), 'r') as f:
        hopping['hardware_config'].append(yaml.load(f,yaml.Loader))
    hopping['energy'].append(pd.read_csv(os.path.join(path,'energy.csv')))
    hop_files = []
    for filename in os.listdir(path):
        if filename[:3] == 'hop':
            idx = int(filename.split('_')[1].split('.')[0])
            hop_files.append((idx, filename))
    hop_files = sorted(hop_files, key = lambda args: args[0])
    dataframes = []
    for idx, filename in hop_files:
        input_df = pd.read_csv(os.path.join(path, filename))
        output_df = pd.DataFrame(columns=columns)
        for key, map in column_map.items():
            new_key, new_data = map(input_df[key].to_numpy())
            output_df[new_key] = new_data
        # output_df = pd.DataFrame(output_df[output_df['mode']==2])
        # output_df = output_df[output_df['mode']==2]
        smoothed_df = smooth_dynamics(output_df, deg, window)

        # unpacking experiment parameters
        experiment_config = hopping['experiment_config'][-1]
        hardware_config = hopping['hardware_config'][-1]
        kin_params = KinematicParameters(**hardware_config['kinematics'])
        mass = experiment_config['dynamics_params']['m']
        pulse = experiment_config['controller']['servo_pos']
        gear_ratio = hardware_config['servo']['gear_ratio']
        psi_deg = pulse_to_angle(pulse, gear_ratio)

        # evaluate FK
        x = smoothed_df['x'].to_numpy()
        FK = forward_kinematics(kin_params,x,jacobian=True,hessian=True)
        smoothed_df['y'] = FK[0][0]
        smoothed_df['l'] = FK[0][1]
        smoothed_df['dydx'] = FK[1][0]
        smoothed_df['dldx'] = FK[1][1]
        smoothed_df['d2yd2x'] = FK[2][0]
        smoothed_df['d2ld2x'] = FK[2][1]
        smoothed_df['psi'] = (np.pi/180)*psi_deg*np.ones(len(smoothed_df))
        smoothed_df['m'] = mass*np.ones(len(smoothed_df))
        smoothed_df['mode'] = output_df['mode'].to_numpy()[window//2:-window//2]
        # dataframes.append(smoothed_df) 
        dataframes.append(smoothed_df[smoothed_df['mode']==2]) 
    hopping['data'].append(dataframes)

In [4]:
from numpy.random import default_rng
N_CHAR_ROWS = sum(len(df) for df in characterization['data'])
N_HOP_ROWS = sum(len(df) for trial in hopping['data'] for df in trial)
ALL_DATA = np.vstack([df.to_numpy() for df in characterization['data']]+[df.to_numpy() for trial in hopping['data'] for df in trial])
CHAR_ROW_INDICES = range(N_CHAR_ROWS)
HOP_ROW_INDICES = range(N_CHAR_ROWS,N_CHAR_ROWS+N_HOP_ROWS)

rng = default_rng()
CHAR_TRAIN_INDICES = rng.choice(CHAR_ROW_INDICES, 50*N_CHAR_ROWS//100, replace=False)
HOP_TRAIN_INDICES = rng.choice(HOP_ROW_INDICES, 50*N_HOP_ROWS//100, replace=False)
# TRAIN_INDICES = list(CHAR_TRAIN_INDICES) + list(HOP_TRAIN_INDICES)
# TRAIN_INDICES = list(CHAR_TRAIN_INDICES)
TRAIN_INDICES = list(HOP_TRAIN_INDICES)
DATA_COLUMN_MAP = {columns[i] : i for i in range(len(columns))}
UNIQUE_PSI = np.sort(np.unique(ALL_DATA[TRAIN_INDICES,DATA_COLUMN_MAP['psi']]))

In [5]:
print(len(TRAIN_INDICES))

4348


In [6]:
from hsa_hopper.hsa_model import HSAModel, LinearModel
def make_lstsq_data(model, row_indices):
    N = model.num_params()
    M = len(row_indices)
    A = np.zeros((M,3+N))
    b = np.zeros(M)
    for i, idx in enumerate(row_indices):
        x = ALL_DATA[idx,DATA_COLUMN_MAP['x']]
        y = ALL_DATA[idx,DATA_COLUMN_MAP['y']]
        l = ALL_DATA[idx,DATA_COLUMN_MAP['l']]
        xdot = ALL_DATA[idx,DATA_COLUMN_MAP['xdot']]
        xddot = ALL_DATA[idx,DATA_COLUMN_MAP['xddot']]
        tau = ALL_DATA[idx,DATA_COLUMN_MAP['tau']]
        dydx = ALL_DATA[idx,DATA_COLUMN_MAP['dydx']]
        dldx = ALL_DATA[idx,DATA_COLUMN_MAP['dldx']]
        d2yd2x = ALL_DATA[idx,DATA_COLUMN_MAP['d2yd2x']]
        d2ld2x = ALL_DATA[idx,DATA_COLUMN_MAP['d2ld2x']]
        psi = ALL_DATA[idx,DATA_COLUMN_MAP['psi']]
        mode = ALL_DATA[idx,DATA_COLUMN_MAP['mode']]
        m = ALL_DATA[idx,DATA_COLUMN_MAP['m']]
        ydot = dydx*xdot
        ldot = dldx*xdot
        if mode == 0:
            b[i] = tau
            A[i,0] = xddot*dldx**2 + ldot*d2ld2x*xdot
            # A[i,0] = 0
            A[i,2] = 0
        elif mode == 2:
            # b[i] = tau - dydx*m*9.81 - m*xddot*dydx**2 - m*ydot*d2yd2x*xdot
            # b[i] = tau - dydx*m*9.81  - m*ydot*d2yd2x*xdot
            # b[i] = tau - dydx*m*9.81 - m*xddot*dydx**2
            b[i] = tau - dydx*m*9.81
            # A[i,0] = -dydx*9.81
            A[i,2] = xdot
            A[i,0] = 0
        A[i,1] = xddot
        z = np.array([l,psi])
        zdot = np.array([ldot,0])
        if model.linear is None:
            N_c = model.w_c.shape[0]
            N_d = model.w_d.shape[0]
            for j in range(N_c):
                A[i,3+j] = dldx*model.dkc(z,j)[0]
            for j in range(N_d):
                A[i,3+N_c+j] = dldx*model.dkd(z,zdot,j)[0]
        else:
            A[i,3] = dldx*l
            A[i,4] = dldx
            A[i,5] = dldx*ldot
            N_c = model.w_c.shape[0]
            N_d = model.w_d.shape[0]
            for j in range(N_c):
                A[i,6+j] = dldx*model.dkc(z,j)[0]
            for j in range(N_d):
                A[i,6+N_c+j] = dldx*model.dkd(z,zdot,j)[0]
    return A,b

In [11]:
"""
Experimental - fitting two potential functions and blending them together with a logistic function
"""

from scipy.optimize import minimize
from scipy.optimize import LinearConstraint
from hsa_hopper.hsa_model import HSAModel

# function to do the model optimization
def governing_equation_lstsq(model):
    N = model.num_params()
    A,b = make_lstsq_data(model,TRAIN_INDICES)

    # rescaling A matrix for better conditioning of constraint evaluation
    SCALE = np.ones(A.shape[1])
    SCALE[0] = 1/10 # leg mass rescaling
    SCALE[1] = 1/1000 # motor inertia rescaling
    SCALE[2] = 1/1000 # motor damping rescaling

    A_scaled = A*SCALE
    ATA = A_scaled.T@A_scaled
    ATb = A_scaled.T@b
    bTb = np.dot(b,b)
    f = lambda x: x.T@ATA@x+bTb-2*x.T@ATb
    jac = lambda x: 2*ATA@x-2*ATb
    # hess = lambda x: 2*ATA

    lb = np.array([1,4,1])
    ub = np.array([5,10,150])
    C = np.eye(A.shape[1])
    if model.linear is not None:
        lb = np.hstack((lb,np.array([0,-1000,0])))
        ub = np.hstack((ub,np.array([2000,1000,100])))
    lb = np.hstack((lb, -100*np.ones(model.w_c.shape[0]), -100*np.ones(model.w_d.shape[0])))
    ub = np.hstack((ub, 100*np.ones(model.w_c.shape[0]), 100*np.ones(model.w_d.shape[0])))
    constraints = [LinearConstraint(C,lb=lb,ub=ub)]

    x0 = (lb+ub)/2
    result = minimize(f, x0, 
                    jac=jac, 
                    constraints=constraints, 
                    method='SLSQP',
                    options={'maxiter' : 10000}
    )
    result.A = A
    result.b = b
    result.sample_size = b.shape[0]
    result.num_params = model.num_params()
    result.m_l = result.x[0]*SCALE[0]
    result.J_x = result.x[1]*SCALE[1]
    result.b_x = result.x[2]*SCALE[2]
    offset = 3
    if model.linear is not None:
        model.linear.k = result.x[offset+0]
        model.linear.f = result.x[offset+1]
        model.linear.b = result.x[offset+2]
        offset += 3
    N_c = model.w_c.shape[0]
    model.w_c = result.x[offset:offset+N_c]
    model.w_d = result.x[offset+N_c:]
    result.model = model
    result.r_sqr = 1-result.fun/np.sum((b-np.average(b))**2)
    result.r_sqr_adj = 1-(1-result.r_sqr)*(result.sample_size-1)/(result.sample_size-result.num_params-1)
    result.mse = result.fun/result.sample_size
    result.rmse = np.sqrt(result.mse)
    result.bic = result.sample_size*np.log(result.mse)+result.num_params*np.log(result.sample_size)
    result.errors = A@(SCALE*result.x)-b
    return result 
    

In [12]:
linear_model = LinearModel(0,0,0)
y_c = y_d = np.zeros((0,0))
w_c = w_d = np.zeros(0)
model = HSAModel(w_c,y_c,w_d,y_d,1,1,linear_model)
linear_result = governing_equation_lstsq(model)
print(linear_result)
print(linear_result.model.linear.attribute_dict())
with open('linear_model.yaml', 'w') as f:
    yaml.dump(linear_result.model.linear.attribute_dict(), f, yaml.Dumper)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 441.9185978746027
           x: [ 3.000e+00  4.000e+00  1.500e+02  1.325e+03 -1.995e+02
                1.051e+01]
         nit: 16
         jac: [ 0.000e+00  1.373e+02 -1.538e-01  2.786e-05 -1.979e-05
                2.301e-04]
        nfev: 19
        njev: 16
           A: [[ 0.000e+00 -4.581e+01 ...  7.072e-02  1.064e-02]
               [ 0.000e+00 -1.046e+02 ...  6.790e-02  1.205e-03]
               ...
               [ 0.000e+00 -1.147e+02 ...  7.052e-02 -9.771e-03]
               [ 0.000e+00 -5.347e+01 ...  7.448e-02 -4.017e-02]]
           b: [ 1.957e+00  1.807e+00 ...  4.526e-01 -1.880e+00]
 sample_size: 4348
  num_params: 3
         m_l: 0.30000000000000004
         J_x: 0.004000000000000149
         b_x: 0.14999999999997515
       model: <hsa_hopper.hsa_model.HSAModel object at 0x000002D718FE6590>
       r_sqr: 0.9640753283205208
   r_sqr_adj: 0.9640505184643886
         mse: 

In [13]:
linear_model = linear_result.model.linear
l0 = -linear_model.f/linear_model.k

In [14]:
print(np.max(ALL_DATA[HOP_TRAIN_INDICES,DATA_COLUMN_MAP['l']]))

0.172176789072045
